# Personality Prediction - Data Cleaning

## Step 1: Import Required Libraries

In [1]:
import pandas as pd
import numpy as np


## Step 2 — Load the dataset

In [2]:
df = pd.read_csv("../data/raw/data-final.csv",sep="\t")

print("Dataset shape:",df.shape)

Dataset shape: (1015341, 110)


## Step 3: Check Missing Values

In [3]:
missing_values = df.isnull().sum()

print("Total missing values:",missing_values.sum())

Total missing values: 186358


In [4]:
missing_values=missing_values[missing_values > 0]

print(missing_values)

EXT1           1783
EXT2           1783
EXT3           1783
EXT4           1783
EXT5           1783
               ... 
screenw        2066
screenh        2066
introelapse    2066
testelapse     1783
country          77
Length: 105, dtype: int64


## Step 4: Check Duplicate Rows

In [5]:
print("Number of duplicates rows:", df.duplicated().sum())

Number of duplicates rows: 0


In [8]:
df.shape

(1015341, 110)

In [9]:
df.isnull().sum().sum()

np.int64(186358)

In [10]:
df.duplicated().sum()

np.int64(0)

In [11]:
df.columns.tolist()

['EXT1',
 'EXT2',
 'EXT3',
 'EXT4',
 'EXT5',
 'EXT6',
 'EXT7',
 'EXT8',
 'EXT9',
 'EXT10',
 'EST1',
 'EST2',
 'EST3',
 'EST4',
 'EST5',
 'EST6',
 'EST7',
 'EST8',
 'EST9',
 'EST10',
 'AGR1',
 'AGR2',
 'AGR3',
 'AGR4',
 'AGR5',
 'AGR6',
 'AGR7',
 'AGR8',
 'AGR9',
 'AGR10',
 'CSN1',
 'CSN2',
 'CSN3',
 'CSN4',
 'CSN5',
 'CSN6',
 'CSN7',
 'CSN8',
 'CSN9',
 'CSN10',
 'OPN1',
 'OPN2',
 'OPN3',
 'OPN4',
 'OPN5',
 'OPN6',
 'OPN7',
 'OPN8',
 'OPN9',
 'OPN10',
 'EXT1_E',
 'EXT2_E',
 'EXT3_E',
 'EXT4_E',
 'EXT5_E',
 'EXT6_E',
 'EXT7_E',
 'EXT8_E',
 'EXT9_E',
 'EXT10_E',
 'EST1_E',
 'EST2_E',
 'EST3_E',
 'EST4_E',
 'EST5_E',
 'EST6_E',
 'EST7_E',
 'EST8_E',
 'EST9_E',
 'EST10_E',
 'AGR1_E',
 'AGR2_E',
 'AGR3_E',
 'AGR4_E',
 'AGR5_E',
 'AGR6_E',
 'AGR7_E',
 'AGR8_E',
 'AGR9_E',
 'AGR10_E',
 'CSN1_E',
 'CSN2_E',
 'CSN3_E',
 'CSN4_E',
 'CSN5_E',
 'CSN6_E',
 'CSN7_E',
 'CSN8_E',
 'CSN9_E',
 'CSN10_E',
 'OPN1_E',
 'OPN2_E',
 'OPN3_E',
 'OPN4_E',
 'OPN5_E',
 'OPN6_E',
 'OPN7_E',
 'OPN8_E',
 'OPN9_E',
 '

## Step 5: Check Data Types

In [12]:
print(df.dtypes.value_counts())

float64    104
str          4
int64        2
Name: count, dtype: int64


In [13]:
non_numeric_columns = df.select_dtypes(exclude=np.number).columns

print("Noon-numeric columns:")
print(non_numeric_columns)

Noon-numeric columns:
Index(['dateload', 'country', 'lat_appx_lots_of_err', 'long_appx_lots_of_err'], dtype='str')


## Handle Invalid Likert Values(0-->NAN)

In [32]:
personality_columns = [f'{t}{i}' for t in ['EXT','EST','AGR','CSN','OPN'] for i in range(1,11)]

df[personality_columns]= df[personality_columns].replace(0, np.nan)

zeros_per_row = df[personality_columns].isna().sum(axis=1)
print(zeros_per_row.value_counts().sort_index().head(15))

0     876217
1     106937
2      19618
3       4911
4       1620
5        670
6        329
7        169
8         98
9         76
10        62
11        44
12        52
13        45
14        40
Name: count, dtype: int64


In [33]:
df = df[zeros_per_row <= 3]

## Deduplicate by IP

In [34]:
print(df['IPC'].value_counts().head())
df = df[df['IPC']==1]

IPC
1    692069
2    104920
3     33988
4     17180
5     11028
Name: count, dtype: int64


## Filter Respons-time Outliers


In [35]:
print(df['testelapse'].describe())
df = df[(df['testelapse'] > 30) & (df['testelapse'] < 3600)]

count    6.909280e+05
mean     6.461845e+02
std      1.562739e+04
min      1.000000e+00
25%      1.700000e+02
50%      2.210000e+02
75%      3.060000e+02
max      5.372971e+06
Name: testelapse, dtype: float64


## Clean Metadata Columns

In [36]:
df = df[df['country'] != 'NONE']
df = df[(df['screenw'] > 100) & (df['screenh'] > 100)]  # drop impossible screen sizes

## Reverse-score negatively keyed items

In [37]:
reverse = {
    'EXT': [2,4,6,8,10],
    'EST' : [2,4],
    'AGR' : [1,3,5,7],
    'CSN' : [2,4,6,8],
    'OPN' : [2,4,6],
   }
for trait, items in reverse.items():
    for i in items:
        df[f'{trait}{i}'] = 6 - df[f'{trait}{i}']

## Impute any remaining NaNs(row-mean per trait)

In [38]:
for trait in ['EXT','EST','AGR','CSN','OPN']:
    cols = [f'{trait}{i}' for i in range(1,11)]
    row_means = df[cols].mean(axis=1)
    df[cols] = df[cols].apply(lambda c: c.fillna(row_means))

## Compute trait Scores

In [39]:
for trait in ['EXT','EST','AGR','CSN','OPN']:
    cols = [f'{trait}{i}' for i in range(1,11)]
    df[f'{trait}_score'] = df[cols].mean(axis=1)

/tmp/ipykernel_4936/3861686089.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'{trait}_score'] = df[cols].mean(axis=1)
/tmp/ipykernel_4936/3861686089.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'{trait}_score'] = df[cols].mean(axis=1)
/tmp/ipykernel_4936/3861686089.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented f

## Validate with Cronbach's alpha

In [41]:
from pingouin import cronbach_alpha
for trait in ['EXT','EST','AGR','CSN','OPN']:
    cols=[f'{trait}{i}' for i in range(1,11)]
    print(trait, cronbach_alpha(df[cols])[0])

EXT 0.8967320514633619
EST 0.873031194274396
AGR 0.8418890417394754
CSN 0.8220336580492744
OPN 0.7990488528643547


## Save Cleaned data

In [42]:
df.to_csv('../data/processed/cleaned_personality_data.csv', index=False)
print(df.shape)

(672676, 115)
